# 호출 수·비용·종료 조건 비교

같은 과제를 세 위임 구조로 처리하면 모델 호출 수와 추정 비용, 종료 조건이 어떻게 달라질까요?


## 실행 환경 준비

Python 3.12와 고정된 패키지 환경을 사용합니다. 키 값은 출력하지 않고 존재 여부만 확인합니다.


In [1]:
# 고정한 패키지를 설치하고 비밀 키의 존재 여부만 확인합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3
from pathlib import Path
import os
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break

MODEL = "gpt-5.4-mini"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print({"langchain": version("langchain"), "langgraph": version("langgraph"), "model": MODEL, "cost_limit_usd": 1.0})


Note: you may need to restart the kernel to use updated packages.
{'langchain': '1.4.0', 'langgraph': '1.2.11', 'model': 'gpt-5.4-mini', 'cost_limit_usd': 1.0}


## 호출 계측기 준비

### 코드 읽기와 비교 경계

`records`는 모델 응답마다 구조 이름, 역할, 입력·출력 토큰을 남기는 관찰 장부입니다. `measured_call()`은 모델 호출과 기록을 한 경계에 묶어, 어떤 구조에서 호출이 생겼는지 빠뜨리지 않게 합니다. 반면 `sub_done`, handoff의 완료 상태, router의 결과 수는 업무가 끝났는지를 판정하는 별도 값입니다. 호출이 오류 없이 반환되었다는 사실과 필요한 결과가 모두 갖춰졌다는 사실은 같지 않습니다.

세 구조는 같은 사용자 문장을 받지만 수행하는 모델 호출의 역할이 다르므로 호출 수만으로 품질을 비교할 수 없습니다. 서브에이전트는 중앙 종합 호출이 추가될 수 있고, 핸드오프는 담당자가 바뀐 뒤 여러 턴을 이어 갈 수 있으며, 라우터는 분류 호출 뒤 독립 전문가를 병렬로 실행합니다. 운영에서는 ‘최대 세 번 호출’ 같은 실행 한도와 ‘정책·배송 결과와 최종 답이 존재함’ 같은 완료 조건을 함께 둡니다. 한도만 두면 불완전한 답도 정상 종료처럼 보일 수 있습니다.


In [2]:
# 세 구조가 함께 사용할 모델, 공식 단가, 기록 목록을 준비합니다.
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

PRICE_PER_M_INPUT = 0.75
PRICE_PER_M_CACHED = 0.075
PRICE_PER_M_OUTPUT = 4.50
model = init_chat_model(f"openai:{MODEL}", reasoning_effort="none")
records: list[dict[str, object]] = []
task = "배송 지연과 환불 가능성을 확인하고 고객 안내를 세 문장으로 작성하세요."


## 응답 사용량 기록 함수


In [3]:
# 일반 입력, 캐시 입력, 출력 토큰을 나누어 비용을 계산합니다.
def record_usage(pattern: str, role: str, response: object) -> None:
    usage = getattr(response, "usage_metadata", None) or {}
    input_tokens = usage.get("input_tokens", 0)
    output_tokens = usage.get("output_tokens", 0)
    details = usage.get("input_token_details", {}) or {}
    cached_tokens = details.get("cache_read", 0)
    uncached_tokens = input_tokens - cached_tokens
    cost = (uncached_tokens * PRICE_PER_M_INPUT + cached_tokens * PRICE_PER_M_CACHED + output_tokens * PRICE_PER_M_OUTPUT) / 1_000_000
    records.append({"pattern": pattern, "role": role, "input": input_tokens, "cached": cached_tokens, "output": output_tokens, "cost_usd": cost})

def measured_call(pattern: str, role: str, prompt: str) -> str:
    response = model.invoke(prompt)
    record_usage(pattern, role, response)
    return response.text


## Subagent 구조 실행

Subagent의 종료 값은 전문 도구가 반환한 업무 결과입니다. `records`의 호출 수나 토큰 수와 서로 다른 층의 값입니다.


In [4]:
# create_agent supervisor가 전문 도구를 실제로 선택하게 합니다.
@tool
def policy_tool(question: str) -> str:
    """환불 정책 확인 항목을 반환합니다."""
    return measured_call("subagent", "policy_tool", "환불 확인 항목: " + question)

subvisor = create_agent(model, tools=[policy_tool], system_prompt="환불 요청에는 policy_tool을 반드시 한 번 사용하고 결과를 종합하세요.")
sub_result = subvisor.invoke({"messages": [{"role": "user", "content": task}]})
sub_trace = [call["name"] for message in sub_result["messages"] for call in getattr(message, "tool_calls", [])]
for message in sub_result["messages"]:
    if getattr(message, "usage_metadata", None):
        record_usage("subagent", "supervisor", message)
sub_done = bool(sub_result["messages"][-1].text and sub_trace == ["policy_tool"])
print({"trace": sub_trace, "done": sub_done})
assert sub_done


{'trace': ['policy_tool'], 'done': True}


## Handoff 노드와 분기


In [5]:
# active_agent를 읽는 조건 함수가 실제 다음 노드를 선택하게 합니다.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class HandoffState(TypedDict):
    active_agent: str
    task: str
    shipping: str
    answer: str
    trace: list[str]

def shipping_node(state: HandoffState) -> dict[str, object]:
    result = measured_call("handoff", "shipping", "배송 확인 항목: " + state["task"])
    return {"shipping": result, "active_agent": "refund", "trace": ["shipping"]}

def refund_node(state: HandoffState) -> dict[str, object]:
    result = measured_call("handoff", "refund", "배송 결과를 이어받아 환불 답변: " + state["shipping"])
    return {"answer": result, "active_agent": "done", "trace": state["trace"] + ["refund"]}

def route_active(state: HandoffState) -> str:
    return state["active_agent"]


## Handoff 그래프 실행


In [6]:
# 조건부 엣지가 shipping과 refund를 차례로 선택하는지 trace로 확인합니다.
builder = StateGraph(HandoffState)
builder.add_node("shipping", shipping_node)
builder.add_node("refund", refund_node)
builder.add_conditional_edges(START, route_active, {"shipping": "shipping"})
builder.add_conditional_edges("shipping", route_active, {"refund": "refund"})
builder.add_edge("refund", END)
handoff_graph = builder.compile()
handoff_result = handoff_graph.invoke({"active_agent": "shipping", "task": task, "shipping": "", "answer": "", "trace": []})
handoff_done = bool(handoff_result["answer"] and handoff_result["active_agent"] == "done")
print({"trace": handoff_result["trace"], "active_agent": handoff_result["active_agent"]})
assert handoff_done and handoff_result["trace"] == ["shipping", "refund"]


{'trace': ['shipping', 'refund'], 'active_agent': 'done'}


## Router 구조 실행

Router의 분류 1회와 선택된 전문 작업 호출을 구분해 기록합니다. 집계표를 읽을 때는 호출 수가 어떤 역할에서 발생했는지까지 보아야 합니다.


In [7]:
# 구조화된 classifier의 destinations를 그대로 읽어 선택된 경로만 실행합니다.
from typing import Literal
from pydantic import BaseModel

class CompareRoute(BaseModel):
    destinations: list[Literal["policy", "shipping"]]

route_result = model.with_structured_output(CompareRoute, include_raw=True).invoke("요청에 필요한 분야를 모두 고르세요: " + task)
if route_result["parsing_error"] or not route_result["parsed"]:
    raise ValueError("라우터 구조화 출력 파싱 실패")
record_usage("router", "classifier", route_result["raw"])
destinations = route_result["parsed"].destinations
router_trace = ["classifier"]
router_results: list[str] = []
for destination in destinations:
    router_trace.append(destination)
    router_results.append(measured_call("router", destination, f"{destination} 담당자로 답하세요: " + task))
router_answer = measured_call("router", "synthesizer", "전문 결과를 세 문장으로 합치세요: " + " ".join(router_results))
router_trace.append("synthesizer")
router_done = bool(destinations and len(router_results) == len(destinations) and router_answer)
print({"route": destinations, "trace": router_trace, "done": router_done})
assert router_done and set(destinations) == {"policy", "shipping"}


{'route': ['policy', 'shipping'], 'trace': ['classifier', 'policy', 'shipping', 'synthesizer'], 'done': True}


## 호출 수와 비용 집계


In [8]:
# 구조별 호출 수, 입력·출력 토큰, 추정 비용을 합산합니다.
summary: dict[str, dict[str, float]] = {}
for row in records:
    item = summary.setdefault(row["pattern"], {"calls": 0, "input": 0, "cached": 0, "output": 0, "cost_usd": 0.0})
    item["calls"] += 1
    item["input"] += row["input"]
    item["cached"] += row["cached"]
    item["output"] += row["output"]
    item["cost_usd"] += row["cost_usd"]
for name, item in summary.items():
    item["cost_usd"] = round(item["cost_usd"], 6)
    print(name, item)
assert all(item["cost_usd"] < 1 for item in summary.values())


subagent {'calls': 3, 'input': 1253, 'cached': 0, 'output': 1013, 'cost_usd': 0.005498}
handoff {'calls': 2, 'input': 122, 'cached': 0, 'output': 154, 'cost_usd': 0.000785}
router {'calls': 4, 'input': 329, 'cached': 0, 'output': 295, 'cost_usd': 0.001574}


## 종료 조건 검증


In [9]:
# 앞 셀에서 계산한 완료 조건 세 개가 모두 참인지 검사합니다.
completion = {"subagent": sub_done, "handoff": handoff_done, "router": router_done}
for pattern, done in completion.items():
    print(pattern, "완료:", done)
assert all(completion.values())


subagent 완료: True
handoff 완료: True
router 완료: True
